In [3]:
### import necessary packages
import os
import pandas as pd # make sure this is < v3.0 !
import numpy as np  # make sure this is < v2.0 !


In [4]:
### specify path to 5bus data
path_to_5bus = "../gtep/data/5bus/"
sorted(os.listdir(path_to_5bus))


['DAY_AHEAD_load.csv',
 'DAY_AHEAD_renewables.csv',
 'REAL_TIME_load.csv',
 'REAL_TIME_renewables.csv',
 'branch.csv',
 'bus.csv',
 'gen.csv',
 'initial_status.csv',
 'reserves.csv',
 'simulation_objects.csv',
 'timeseries_pointers.csv']

In [3]:
### this is from the Prescient tutorial; I replaced data_path with the relevant path, 
### changed the length of the simulation to run for a full year, and switched contingency monitoring off
from prescient.simulator import Prescient
Prescient().simulate(
        data_path = path_to_5bus,
        input_format = "rts-gmlc",
        simulate_out_of_sample = True, 
        run_sced_with_persistent_forecast_errors = True, 
        output_directory = "5bus_output",
        start_date = "01-01-2020", 
        num_days = 366,              # 2020 is a leap year
        reserve_factor = 0.1, 
        sced_solver = "gurobi", 
        sced_frequency_minutes = 60, # THIS HAS TO BE 60 OR ELSE IT WILL NOT RUN
        sced_horizon = 1, 
        sced_slack_type = "ref-bus-and-branches", 
        ruc_slack_type = "ref-bus-and-branches", 
        ruc_horizon = 24,
        ruc_mipgap = 0.01, 
        deterministic_ruc_solver = "gurobi", 
        #deterministic_ruc_solver_options = {"feas":"off", "DivingF":"on",},    # i don't think this exists anymore...?
        output_solver_logs = False, 
        compute_market_settlements = True, 
        monitor_all_contingencies = False, 
        price_threshold = 1000, 
        contingency_price_threshold = 100, 
        reserve_price_threshold = 5, 
        ruc_network_type = "btheta", # need to include this for it to run
)


Interactive Python mode detected; using default matplotlib backend for plotting.
Initializing simulation...


/home/rmalfan/anaconda3/envs/prescient_new/lib/python3.9/site-packages/egret/parsers/rts_gmlc/parser.py:164: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  metadata_df.loc['Look_Ahead_Periods_per_Step']['DAY_AHEAD'] = 0
/home/rmalfan/anaconda

Dates to simulate: 2020-01-01 to 2020-12-31
RUC activation hours: 0
Final RUC date: 2020-12-31
Using current day's forecasts for RUC solves
Using persistent forecast error model when projecting demand and renewables in SCED


Extracting scenario to simulate

Pyomo model construction time:         0.09

Pyomo model solve time: 0.22385454177856445

Deterministic RUC Cost: 21065.54

Fixed costs:        15649.03
Variable costs:      5416.50


Renewables curtailment summary (time-period, aggregate_quantity):
2020-01-01 00:00        68.02
2020-01-01 01:00        60.43
2020-01-01 02:00        59.32
2020-01-01 03:00        59.21
2020-01-01 04:00        53.53
2020-01-01 05:00        43.80
2020-01-01 06:00        27.86
2020-01-01 07:00        29.26
2020-01-01 08:00        21.31
2020-01-01 22:00        28.27
2020-01-01 23:00        33.64
Solving day-ahead market
Computing day-ahead prices using method ACHP.
Simulating time_step  2020-01-01 00:00

Solving SCED instance
Solving for LMPs
Fixed costs

In [4]:
### print output files from simulation 
sorted(os.listdir("5bus_output"))


['bus_detail.csv',
 'contingency_detail.csv',
 'daily_summary.csv',
 'hourly_gen_summary.csv',
 'hourly_summary.csv',
 'line_detail.csv',
 'overall_simulation_output.csv',
 'plots',
 'renewables_detail.csv',
 'reserves_detail.csv',
 'runtimes.csv',
 'thermal_detail.csv',
 'virtual_detail.csv']

In [ ]:
### find the highest load shed day:

# load in daily summary data 
daily_summary_data = pd.read_csv('5bus_output/daily_summary.csv')
daily_summary_data.head()


,Date,Demand,Renewables available,Renewables used,Renewables penetration rate,Average price,Fixed costs,Generation costs,Load shedding,Over generation,...,Sum nominal ramps,Renewables energy payments,Renewables uplift payments,Thermal energy payments,Thermal uplift payments,Total energy payments,Total uplift payments,Total reserve payments,Total payments,Average payments
0,2020-01-01,1924.263,2209.998,1552.187,417.169342,8.997108,15649.0350,1663.766103,0.0,0.0,...,143.490,12883.458154,0.0,11196.170079,0.0,24079.628234,0.0,6380.469997,30460.098231,15.829488
1,2020-01-02,1905.367,2844.259,1657.367,668.293145,8.233504,15687.8475,0.000000,0.0,0.0,...,0.000,0.000000,0.0,0.000000,0.0,0.000000,0.0,10800.000000,10800.000000,5.668199
2,2020-01-03,1882.903,2612.521,1552.598,470.049802,8.917876,15687.8475,1103.648321,0.0,0.0,...,46.131,5176.754477,0.0,10595.098246,0.0,15771.852722,0.0,7544.627286,23316.480008,12.383261
3,2020-01-04,1824.837,1220.154,954.653,109.707027,13.213532,15687.8475,8424.693942,0.0,0.0,...,131.766,6275.957054,0.0,18276.168196,0.0,24552.125250,0.0,5553.101121,30105.226371,16.497488
4,2020-01-05,1812.049,2797.351,1564.049,630.664919,8.657518,15687.8475,0.000000,0.0,0.0,...,5.323,3034.957310,0.0,8681.355470,0.0,11716.312780,0.0,8724.837074,20441.149854,11.280683


In [15]:
# print the date with the highest load shed 
max_load_shed_idx = daily_summary_data['Load shedding'].idxmax()
max_load_shed_day = daily_summary_data.loc[max_load_shed_idx, 'Date']
print("highest load shed day:", max_load_shed_day)


highest load shed day: 2020-06-30


In [6]:
### find highest transmission congestion day: (i pulled some of this code from the Prescient tutorial)

# load in nominal line flow data
line_detail = pd.read_csv('5bus_output/line_detail.csv')

# load in file with line limits
branch_csv = pd.read_csv(f'{path_to_5bus}/branch.csv', index_col=0)

# TODO: need to confirm 'cont rating' is the correct column for line limits! 
cont_rating = branch_csv['Cont Rating']


In [7]:
# renaming 'Cont Rating' to 'Line' to match index of line_detail
cont_rating.index.name = 'Line'

# making cont_rating a dict
cont_rating = cont_rating.to_dict()
cont_rating


{'branch_2_3': 80.0,
 'branch_1_2': 66.6,
 'branch_1_4': 66.6,
 'branch_4_10': 66.6,
 'branch_1_10': 66.6,
 'branch_3_4_0': 66.6,
 'branch_3_4_1': 28.4}

In [8]:
line_detail.head()


,Date,Hour,Minute,Line,Flow,Violation
0,2020-01-01,0,0,branch_2_3,-10.29879,0.0
1,2020-01-01,0,0,branch_1_2,12.99421,0.0
2,2020-01-01,0,0,branch_1_4,-4.77400,0.0
3,2020-01-01,0,0,branch_4_10,4.02021,0.0
4,2020-01-01,0,0,branch_1_10,-4.02021,0.0


In [9]:
# create new column in line_detail for congestion values (initially just populated by |flow|s)
line_detail['Congestion'] = np.abs(line_detail['Flow'])

# TODO: tests to confirm flows are being divided by correct values to get congestion
for line_name in cont_rating.keys():
    line_detail['Congestion'] = np.where(line_detail['Line'] == line_name, line_detail['Congestion'] / cont_rating[line_name], line_detail['Congestion'])



In [ ]:
# create new dataframe that has daily congestion instead of hourly. TODO: write a test for this also
daily_line_detail = line_detail.groupby('Date')['Congestion'].sum().reset_index()
daily_line_detail.head()


,Date,Congestion
0,2020-01-01,33.197833
1,2020-01-02,34.402963
2,2020-01-03,33.938463
3,2020-01-04,32.874503
4,2020-01-05,32.333207


In [13]:
# print the date with the highest congestion
max_congestion_idx = daily_line_detail['Congestion'].idxmax()
max_congestion_day = daily_line_detail.loc[max_congestion_idx, 'Date']
print("highest congestion day:", max_congestion_day)


highest congestion day: 2020-07-30


Defining seasons as:
- Winter: Dec, Jan, Feb
- Spring: Mar, Apr, May
- Summer: June, July, Aug
- Fall: Sept, Oct, Nov

In [ ]:
### highest load shed in each season

# make sure daily summary data is indexed by Date
daily_summary_data.set_index('Date', inplace=True)


In [ ]:
# range for winter is split
winter_split_range = pd.concat([daily_summary_data.loc['2020-12-01':'2020-12-31'], 
                               daily_summary_data.loc['2020-01-01':'2020-02-29']])

print("winter:", winter_split_range['Load shedding'].idxmax())
print("spring:", daily_summary_data.loc['2020-03-01':'2020-05-31', 'Load shedding'].idxmax())
print("summer:", daily_summary_data.loc['2020-06-01':'2020-08-31', 'Load shedding'].idxmax())
print("fall:", daily_summary_data.loc['2020-09-01':'2020-11-30', 'Load shedding'].idxmax())


Winter: 2020-12-01
spring: 2020-03-01
summer: 2020-06-30
fall: 2020-09-15


In [23]:
### highest congestion in each season

# make sure daily line detail is indexed by Date
daily_line_detail.set_index('Date', inplace=True)


In [25]:
# range for winter is split
winter_split_range_lines = pd.concat([daily_line_detail.loc['2020-12-01':'2020-12-31'], 
                                      daily_line_detail.loc['2020-01-01':'2020-02-29']])

print("winter:", winter_split_range_lines['Congestion'].idxmax())
print("spring:", daily_line_detail.loc['2020-03-01':'2020-05-31', 'Congestion'].idxmax())
print("summer:", daily_line_detail.loc['2020-06-01':'2020-08-31', 'Congestion'].idxmax())
print("fall:", daily_line_detail.loc['2020-09-01':'2020-11-30', 'Congestion'].idxmax())


winter: 2020-01-14
spring: 2020-05-22
summer: 2020-07-30
fall: 2020-09-08


In [26]:
### lowest congestion in each season
print("winter:", winter_split_range_lines['Congestion'].idxmin())
print("spring:", daily_line_detail.loc['2020-03-01':'2020-05-31', 'Congestion'].idxmin())
print("summer:", daily_line_detail.loc['2020-06-01':'2020-08-31', 'Congestion'].idxmin())
print("fall:", daily_line_detail.loc['2020-09-01':'2020-11-30', 'Congestion'].idxmin())


winter: 2020-02-26
spring: 2020-03-01
summer: 2020-06-01
fall: 2020-11-01


In [ ]:
### typical congestion (median day) in each season. TODO: is median the correct metric to use here?
print("winter:", winter_split_range_lines.index[winter_split_range_lines['Congestion'] == winter_split_range_lines['Congestion'].median()])

spring_range = daily_line_detail.loc['2020-03-01':'2020-05-31']
summer_range = daily_line_detail.loc['2020-06-01':'2020-08-31']
fall_range = daily_line_detail.loc['2020-09-01':'2020-11-30']

# have to do some stuff here b/c number of rows are even for some seasons. TODO: make sure this is actually returning the median index
print("spring:", spring_range.index[spring_range['Congestion'] == spring_range['Congestion'].quantile(interpolation='nearest')])
print("summer:", summer_range.index[summer_range['Congestion'] == summer_range['Congestion'].quantile(interpolation='nearest')])
print("fall:", fall_range.index[fall_range['Congestion'] == fall_range['Congestion'].median()])


winter: Index(['2020-01-21'], dtype='object', name='Date')
spring: Index(['2020-05-01'], dtype='object', name='Date')
summer: Index(['2020-06-08'], dtype='object', name='Date')
fall: Index(['2020-09-27'], dtype='object', name='Date')
